In [ ]:
import pandas as pd
import numpy as np
import pyodbc
from sqlalchemy import create_engine
import urllib
import warnings
import joblib
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from imblearn.over_sampling import SMOTE

warnings.filterwarnings('ignore')

In [2]:
# Cấu hình kết nối đến SQL Server
server = 'localhost,1433'
database = 'fintech'
username = 'sa'
password = 'huong6978'

connection_string = (
    r'DRIVER={ODBC Driver 17 for SQL Server};'
    rf'SERVER={server};'
    rf'DATABASE={database};'
    rf'UID={username};'
    rf'PWD={password};'
)
params = urllib.parse.quote_plus(connection_string)
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")

try:
    df = pd.read_sql("SELECT * FROM credit_risk_staging", engine)
    print("Đã nạp thành công dữ liệu từ SQL Server!")
except Exception as e:
    print("Lỗi:", e)

print("Kích thước ban đầu:", df.shape)
df.head()


Đã nạp thành công dữ liệu từ SQL Server!
Kích thước ban đầu: (32581, 10)


,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_amnt,loan_int_rate,loan_percent_income,cb_person_default_on_file,loan_status
0,22,59000,RENT,123.0,PERSONAL,35000,16.02,0.59,True,True
1,21,9600,OWN,5.0,EDUCATION,1000,11.14,0.10,False,False
2,25,9600,MORTGAGE,1.0,MEDICAL,5500,12.87,0.57,False,True
3,23,65500,RENT,4.0,MEDICAL,35000,15.23,0.53,False,True
4,24,54400,RENT,8.0,MEDICAL,35000,14.27,0.55,True,True


In [ ]:
# Loại bỏ Outliers (Trường hợp dữ liệu chưa được lọc ở bước Ingestion)
df = df[df['person_age'] <= 100]
df = df[(df['person_emp_length'] <= 100) | (df['person_emp_length'].isnull())]

# Ánh xạ biến nhị phân cb_person_default_on_file ('t'/'f' thành 1/0)
df['cb_person_default_on_file'] = df['cb_person_default_on_file'].astype(int)
df['loan_status'] = df['loan_status'].astype(int)

print("Kích thước sau khi làm sạch cơ bản:", df.shape)

Kích thước sau khi làm sạch cơ bản: (32574, 10)


In [4]:
# Tách features (X) và target (y)
X = df.drop('loan_status', axis=1)
y = df['loan_status']

# Chia tập dữ liệu 80% train - 20% test, sử dụng stratify=y để giữ nguyên tỉ lệ mất cân bằng ban đầu trong cả 2 tập
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("Kích thước X_train:", X_train.shape)
print("Kích thước X_test:", X_test.shape)

Kích thước X_train: (26059, 9)
Kích thước X_test: (6515, 9)


In [ ]:
num_features = ['person_age', 'person_income', 'person_emp_length', 'loan_amnt', 'loan_int_rate', 'loan_percent_income', 'cb_person_default_on_file']
cat_features = ['person_home_ownership', 'loan_intent']

# Pipeline xử lý cho cột số
num_transformer = Pipeline(steps=[
    ('imputer', IterativeImputer(random_state=42, max_iter=10)),
    ('scaler', StandardScaler())
])

# Pipeline xử lý cho cột phân loại
cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Gộp lại thành một Preprocessor tổng
preprocessor = ColumnTransformer(transformers=[
    ('num', num_transformer, num_features),
    ('cat', cat_transformer, cat_features)
])

# Chỉ 'fit' trên X_train để học các tham số, sau đó 'transform' cho cả X_train và X_test
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

# Lấy lại tên các cột sau khi One-Hot Encoding
ohe_feature_names = preprocessor.named_transformers_['cat'].named_steps['onehot'].get_feature_names_out(cat_features)
feature_names = num_features + list(ohe_feature_names)

X_train_df = pd.DataFrame(X_train_processed, columns=feature_names)
X_test_df = pd.DataFrame(X_test_processed, columns=feature_names)

print("Hoàn tất quá trình Pipeline (X_train_df dimensions):", X_train_df.shape)

Hoàn tất quá trình Pipeline (X_train_df dimensions): (26059, 17)


In [6]:
print("Phân phối y_train TRƯỚC khi SMOTE:\n", y_train.value_counts())

smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_df, y_train)

# XỬ LÝ NÂNG CAO (Thực tế): Binarize lại các cột One-Hot Encoding do SMOTE sinh ra giá trị lẻ
ohe_cols = list(ohe_feature_names)
X_train_resampled[ohe_cols] = np.where(X_train_resampled[ohe_cols] >= 0.5, 1, 0)

print("\nPhân phối y_train SAU khi SMOTE:\n", y_train_resampled.value_counts())
print("Kích thước X_train mới:", X_train_resampled.shape)

Phân phối y_train TRƯỚC khi SMOTE:
 loan_status
0    20373
1     5686
Name: count, dtype: int64

Phân phối y_train SAU khi SMOTE:
 loan_status
0    20373
1    20373
Name: count, dtype: int64
Kích thước X_train mới: (40746, 17)


In [7]:
# 1. Lưu Pipeline tiền xử lý
joblib.dump(preprocessor, 'preprocessor.pkl')
print("Đã lưu mô hình tiền xử lý thành file 'preprocessor.pkl'")

# 2. Gộp feature và target lại
train_final = pd.concat([X_train_resampled.reset_index(drop=True), y_train_resampled.reset_index(drop=True)], axis=1)
test_final = pd.concat([X_test_df.reset_index(drop=True), y_test.reset_index(drop=True)], axis=1)

# Lưu dữ liệu dưới dạng 'Bảng Vàng' trở lại SQL Server
train_final.to_sql("gold_credit_risk_train", engine, if_exists="replace", index=False)
test_final.to_sql("gold_credit_risk_test", engine, if_exists="replace", index=False)
    
print("Đã lưu các 'Bảng Vàng' vào hệ thống thành công!")

# engine quản lý connection pooling tự động


Đã lưu mô hình tiền xử lý thành file 'preprocessor.pkl'
Đã lưu các 'Bảng Vàng' vào hệ thống thành công!
